# Hybrid Multi-Domain Image Forensics — interactive demo
Walks through the pipeline step by step: input → preprocessing → spatial & frequency features → fusion CNN → detection → localization.

In [ ]:
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from forensics.preprocess import load_image, limit_size
from forensics import spatial as sp, frequency as fq
from forensics.features import forensic_maps
from forensics.detector import ForensicDetector

img = limit_size(load_image('../samples/splice.png'))   # 1-2. input + preprocessing
plt.imshow(img); plt.axis('off'); plt.title('input');

## 3a. Spatial-domain features

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (t, m) in zip(ax, [('ELA', sp.ela_visual(img)), ('SRM residual', abs(sp.srm_residuals(img)[1])),
                         ('local noise', sp.local_noise_map(img)), ('edges', sp.edge_map(img))]):
    a.imshow(m, cmap='inferno'); a.set_title(t); a.axis('off')

## 3b. Frequency-domain features (FFT / DCT / DWT)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, (t, m) in zip(ax, [('FFT spectrum', fq.fft_spectrum(img)), ('FFT high-pass', fq.fft_highpass_map(img)),
                         ('DCT AC energy', fq.dct_ac_map(img)), ('JPEG blockiness', fq.blockiness_map(img)),
                         ('wavelet HH noise', fq.wavelet_noise_map(img))]):
    a.imshow(m, cmap='viridis'); a.set_title(t); a.axis('off')

## 4-6. Fusion CNN → detection → localization

In [ ]:
det = ForensicDetector('../checkpoints/hybrid.pt')
res = det.analyze(img)
print(res.label, f'P(manipulated) = {res.probability:.3f}')
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
gt = load_image('../samples/splice_mask.png')
for a, (t, m) in zip(ax, [('input', res.image), ('overlay', res.overlay), ('predicted mask', res.mask), ('ground truth', gt)]):
    a.imshow(m, cmap='gray'); a.set_title(t); a.axis('off')
res.evidence[:3]